First, run `poetry add matplotlib-scalebar`  in your terminal to install one additional package.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

In this notebook, we will explore behavioral data (ear features) from two mice, `B39` and `B54`, 
reacting to a white noise sound stimulus. Each sound stimulus lasts for two seconds and is played 
around ten (+/- 1) times over the course of a session. Ear responses are decomposed into 3D rotations along the 
pitch, roll, and yaw axis. Can you spot a difference between the two mouse's ear responses?

In [ ]:
#Load the Data
B39_ear_responses = pd.read_csv('ear-response-data/B39_Sound_0.csv')
B54_ear_responses = pd.read_csv('ear-response-data/B54_Sound_0.csv')

print(f"B39 Dataframe Dimensions: {B39_ear_responses.shape}")
print(f"B54 Dataframe Dimensions: {B54_ear_responses.shape}")

print("B39:")
B39_ear_responses

The `Sound_0` column contains triggers for when the sound stimulus is presented
to the mouse (shown below). Using these trigger timestamps, we want to extract the ear 
responses from one second before the stimulus is presented, through the two second
stimulus period, and one second after the stimulus is presented.


In [ ]:
B39_ear_responses[B39_ear_responses['Sound_0']]

Now, how would we use this information to produce a list of DataFrame indices for when the trigger occured?

TODO: complete the `trigger_indices=` line to produce a list of indices where Sound_0 is True.

In [ ]:
def index_trials(ear_responses_df,
                pre_trial=100, #1 second before trigger
                trial_len=200, #2 second stimulus
                post_trial=100, #1 second after trigger
                ):
    #TODO get the trigger indices
    trigger_indices = 

    #get the specified range between pre_trial and post_trial
    trial_ranges = np.asarray([trigger_indices - pre_trial, trigger_indices + trial_len + post_trial]).T

    return trial_ranges, trigger_indices

trial_ranges, trigger_indices = index_trials(B54_ear_responses)

print(f"Trial Ranges: {trial_ranges}")
print(f"Trigger Indices: {trigger_indices}" )

Great! Now that we have a function to calculate our trial indices, we can use
these to plot each trial's pitch, yaw, and roll variables together as stacked trace plots.\
We do this by baselining the responses (pretrial mean centered), and plotting each
trial's traces with an offset. Now we can see how movements in the `Pitch`, `Roll`,
and `Yaw` dimensions changes along the same time axis.

In [ ]:
def plot_mouse_ear_traces(ear_responses_df, pre_sound=100, sound_duration=200, post_sound=100, num_sounds=1, mouse_id=None, ears=['left', 'right'], spacing_factor=50):
    #get trial + trigger indices
    trial_indices, trigger_indices = index_trials(ear_responses_df)
    num_trials = trigger_indices.shape[0]
    

    #2 subplots per ear
    fig = plt.figure(figsize=(num_sounds * 8, len(ears) * 4))
    gs = gridspec.GridSpec(num_sounds, len(ears), width_ratios=[1, 1])


    #for each ear
    for ear_num, ear in enumerate(ears):
        ax = plt.subplot(gs[ear_num])

        #for every trial in each ear
        for trial_num, trial_idx in enumerate(trial_indices):
            # print(f"Working on Trial {trial_num} : {trial_idx}")

        #extract ear responses per trial
            trial = ear_responses_df.iloc[trial_idx[0] : trial_idx[1]]
    
            offset = (num_trials - 1 - trial_num) * spacing_factor #decsending order of trials

            #"basline" ear responses to calculate deviation from prestimulus mean
            baselined_p = trial[f'{ear}_pitch'] - np.nanmean(trial[f'{ear}_pitch'][:pre_sound])
            baselined_r = trial[f'{ear}_roll'] - np.nanmean(trial[f'{ear}_roll'][:pre_sound])
            baselined_y = trial[f'{ear}_yaw'] - np.nanmean(trial[f'{ear}_yaw'][:pre_sound])
            
            #plot traces with offset
            x_axis = (np.arange(-pre_sound, sound_duration + post_sound) / 100)
            ax.plot(x_axis, baselined_p + offset, color='red', label='Pitch', alpha=0.5)
            ax.plot(x_axis, baselined_r + offset, color='green', label='Roll', alpha=0.5)
            ax.plot(x_axis, baselined_y + offset, color='blue', label='Yaw', alpha=0.5)

    
        y_min = - spacing_factor
        y_max = (num_trials- 1) * spacing_factor + spacing_factor

        #formatting
        ax.set_ylim(y_min, y_max)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Trial #") 
        ax.set_xlim(-pre_sound / 100, (sound_duration + post_sound) / 100)
        y_ticks = [(num_trials - 1 - i) * spacing_factor for i in range(num_trials)]
        ax.set_yticks(y_ticks)
        ax.set_yticklabels(np.arange(num_trials))


        #plot when the trial starts, and when it ends
        ax.axvline(x=0, color="black", linestyle="--", alpha=0.5)
        ax.axvline(x=sound_duration / 100, color="black", linestyle=":", alpha=0.5)

        #set titles for each subplot
        ear_labels = ['Left', 'Right']
        ax.set_title(f"{ear_labels[ear_num]} Ear Rotation")

        scalebar = ScaleBar(
        dx=1,
        units="deg",
        dimension="angle",
        fixed_value=5,
        width_fraction=0.01,
        location="lower right",
        scale_loc="right",
        rotation="vertical-only",
        color="grey",
        box_alpha=0.0)

        ax.add_artist(scalebar)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[:3], labels[:3], loc='upper right', prop={'size': 8}, framealpha=0.0)
        

    #set figure title
    if mouse_id is not None:
        plt.suptitle(f"Mouse {mouse_id} Ear Response")
    else:
        plt.suptitle(f"Ear Response")
        
        plt.show()

plot_mouse_ear_traces(B39_ear_responses, mouse_id='B39')
plot_mouse_ear_traces(B54_ear_responses, mouse_id='B54')

Did you spot a difference? Let's see if this difference is visible when we compare the mean response across trials

TODO: Calculate the mean and standard error of each `all_traces_x` variable.

In [ ]:
def plot_mean_mouse_ear_traces(ear_responses_df, pre_sound=100, sound_duration=200, post_sound=100, num_sounds=1, mouse_id=None, ears=['left', 'right'], spacing_factor=50):
    
    #get trial + trigger indices
    trial_indices, trigger_indices = index_trials(ear_responses_df)
    num_trials = trigger_indices.shape[0]
    
    #2 subplots per ear
    fig = plt.figure(figsize=(num_sounds * 10, len(ears) * 2))
    gs = gridspec.GridSpec(num_sounds, len(ears), width_ratios=[1, 1])


    #for each ear
    for ear_num, ear in enumerate(ears):
        ax = plt.subplot(gs[ear_num])

        #bins:
        all_traces_p = []
        all_traces_r = []
        all_traces_y = []

        #for every trial in each ear
        for trial_num, trial_idx in enumerate(trial_indices):
            # print(f"Working on Trial {trial_num} : {trial_idx}")

            #extract ear responses per trial
            trial = ear_responses_df.iloc[trial_idx[0] : trial_idx[1]]
    
            offset = (num_trials - 1 - trial_num) * spacing_factor #decsending order of trials

            #"basline" ear responses to calculate deviation from prestimulus mean, but instead of plotting them, we bin!
            baselined_p = trial[f'{ear}_pitch'] - np.nanmean(trial[f'{ear}_pitch'][:pre_sound])
            all_traces_p.append(baselined_p)

            baselined_r = trial[f'{ear}_roll'] - np.nanmean(trial[f'{ear}_roll'][:pre_sound])
            all_traces_r.append(baselined_r)

            baselined_y = trial[f'{ear}_yaw'] - np.nanmean(trial[f'{ear}_yaw'][:pre_sound])
            all_traces_y.append(baselined_y)



            
            #plot traces with offset
            x_axis = (np.arange(-pre_sound, sound_duration + post_sound) / 100)

        #TODO calculate the means of the binned baselined traces
        meaned_p = 
        meaned_r = 
        meaned_y = 

        #TODO calculate the standard error of the mean per binned baselined trace
        ste_p = 
        ste_r = 
        ste_y = 

        #plot mean traces
        ax.plot(x_axis, meaned_p + offset, color='red', label='Pitch', alpha=0.5)
        ax.plot(x_axis, meaned_r + offset, color='green', label='Roll', alpha=0.5)
        ax.plot(x_axis, meaned_y + offset, color='blue', label='Yaw', alpha=0.5)
        
        #fill shaded area where standard error is
        ax.fill_between(x_axis, meaned_p - ste_p + offset, meaned_p + ste_p + offset,
                        color='red', alpha=0.20)
        ax.fill_between(x_axis, meaned_r - ste_r + offset, meaned_r + ste_r + offset,
                        color='green', alpha=0.20)
        ax.fill_between(x_axis, meaned_y - ste_y + offset, meaned_y + ste_y + offset,
                        color='blue', alpha=0.20)

        #formatting
        ax.set_ylim(-50, 50)
        ax.set_xlabel("Time (seconds)")
        ax.set_ylabel("Change from Mean (Deg)") 
        ax.set_xlim(-pre_sound / 100, (sound_duration + post_sound) / 100)

         #plot when the trial starts, and when it ends
        ax.axvline(x=0, color="black", linestyle="--", alpha=0.5)
        ax.axvline(x=sound_duration / 100, color="black", linestyle=":", alpha=0.5)

        #set titles for each subplot
        ear_labels = ['Left', 'Right']
        ax.set_title(f"{ear_labels[ear_num]} Ear Rotation")
        ax.legend()


    #set figure title
    if mouse_id is not None:
        plt.suptitle(f"Mouse {mouse_id} Ear Response")
    else:
        plt.suptitle(f"Ear Response")
        
        plt.show()

plot_mean_mouse_ear_traces(B39_ear_responses, mouse_id="B39")
plot_mean_mouse_ear_traces(B54_ear_responses, mouse_id="B54")